### Tools

Models can request to call tools that perform tasks such as fetching data from a databases, searching the web, or running code. Tools are pairings of: 

1.A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)

2.A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots don’t “talk” in the way humans use language; they **mimic** sounds they hear, including human speech. Their ability to do this comes from a mix of anatomy, social behavior, and cognition:\n\n| Why parrots can mimic speech | How it works |\n|------------------------------|--------------|\n| **Vocal anatomy** – a highly flexible syrinx (the bird’s voice box) and a muscular tongue/soft palate let them produce a wide range of sounds, including the frequencies of human vowels and consonants. | Air passes through the syrinx; tiny adjustments of the membranes and muscles shape the sound. |\n| **Social learning** – parrots are naturally **social, flock‑oriented birds** that rely on vocal communication to maintain bonds, identify individuals, and coordinate activities. | In the wild they copy calls of mates, parents, and flock members. In captivity, they treat a human caretaker as part of their “flock” and copy the most frequent sounds they hear. |\n| **Cognitive flex

In [6]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [7]:

response = model_with_tools.invoke(
    "What's the weather like in Boston?"
)

print(response)

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


content='' additional_kwargs={'reasoning_content': 'User asks: "What\'s the weather like in Boston?" We need to call get_weather function with location "Boston".', 'tool_calls': [{'id': 'fc_213a6044-335e-443e-b14d-e8b149d401b7', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 127, 'total_tokens': 178, 'completion_time': 0.108906422, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.004896677, 'prompt_tokens_details': None, 'queue_time': 0.211337631, 'total_time': 0.113803099}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a655021f7d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a13a20-9c64-7353-bdb3-e96ada1a9521-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_213a6044-335e-443e-b14d-e8b149d401b7', 'type': 'tool_call'}]

### Tool Execution Loop

In [8]:
# Step 1: Model generates tool calls
messages = [{"role":"user","content":"What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# The current weather in Boston is 72F and sunny.

It’s sunny in Boston right now.


In [9]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks for weather in Boston. We have function get_weather. Use it.', 'tool_calls': [{'id': 'fc_e77a1655-178c-4022-a815-c391d3891777', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 126, 'total_tokens': 171, 'completion_time': 0.095300573, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.006949299, 'prompt_tokens_details': None, 'queue_time': 0.333253746, 'total_time': 0.102249872}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_6fb4303df4', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a13a2e-a719-79f1-8247-5d4ac10df732-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_e77a1655-178c-402